In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
df_user=pd.read_csv("raw/profile/USER_DATA_FINAL.csv", encoding='utf-8-sig')
df_job=pd.read_csv("processed/COMBINED_DATA_PROCESSED.csv", encoding='utf-8-sig')

In [12]:
provinces_list = [
    'Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ',
    'An Giang', 'Bà Rịa - Vũng Tàu', 'Bắc Giang', 'Bắc Kạn', 'Bạc Liêu',
    'Bắc Ninh', 'Bến Tre', 'Bình Định', 'Bình Dương', 'Bình Phước',
    'Bình Thuận', 'Cà Mau', 'Cao Bằng', 'Đắk Lắk', 'Đắk Nông',
    'Điện Biên', 'Đồng Nai', 'Đồng Tháp', 'Gia Lai', 'Hà Giang',
    'Hà Nam', 'Hà Tĩnh', 'Hải Dương', 'Hậu Giang', 'Hòa Bình',
    'Hưng Yên', 'Khánh Hòa', 'Kiên Giang', 'Kon Tum', 'Lai Châu',
    'Lâm Đồng', 'Lạng Sơn', 'Lào Cai', 'Long An', 'Nam Định',
    'Nghệ An', 'Ninh Bình', 'Ninh Thuận', 'Phú Thọ', 'Quảng Bình',
    'Quảng Nam', 'Quảng Ngãi', 'Quảng Ninh', 'Quảng Trị', 'Sóc Trăng',
    'Sơn La', 'Tây Ninh', 'Thái Bình', 'Thái Nguyên', 'Thanh Hóa',
    'Thừa Thiên Huế', 'Tiền Giang', 'Trà Vinh', 'Tuyên Quang', 'Vĩnh Long',
    'Vĩnh Phúc', 'Yên Bái', 'Phú Yên'
]

# 2. Bảng map các tên viết tắt (Dạng Dictionary)
province_aliases = {
    'TP.HCM': 'Hồ Chí Minh',
    'TP HCM': 'Hồ Chí Minh',
    'Sài Gòn': 'Hồ Chí Minh',
    'HCMC': 'Hồ Chí Minh',
    'Vũng Tàu': 'Bà Rịa - Vũng Tàu',
    'Huế': 'Thừa Thiên Huế',
    'Dak Lak': 'Đắk Lắk',
}

def extract_province(addr):
    if pd.isna(addr): return 'Khác'
    
    addr_str = str(addr).lower()
    
    # Bước 1: Kiểm tra Alias (Dùng .items() cho Dictionary)
    for alias, formal in province_aliases.items():
        if alias.lower() in addr_str:
            return formal
            
    # Bước 2: Kiểm tra 63 tỉnh (Dùng vòng lặp cho List)
    for p in provinces_list:
        if p.lower() in addr_str:
            return p
            
    # Bước 3: Các trường hợp đặc thù trong tuyển dụng
    if any(k in addr_str for k in ['toàn quốc', 'tất cả', 'linh hoạt']):
        return 'Toàn quốc'
    if any(k in addr_str for k in ['nước ngoài', 'singapore', 'japan', 'usa', 'nhật bản']):
        return 'Nước ngoài'
    if any(k in addr_str for k in ['tại nhà', 'làm việc từ xa', 'remote']):
        return 'Tại Nhà'
        
    return 'Khác'

In [13]:
def map_industry_group(main_industry):
    if pd.isna(main_industry):
        return 'Khác'
    
    text = str(main_industry).lower()
    
    # 1. Công nghệ thông tin
    if any(k in text for k in ['it', 'cntt', 'phần mềm', 'phần cứng', 'lập trình', 'developer', 'mạng', 'tester', 'seo']):
        return 'Công nghệ thông tin'
    
    # 2. Tài chính - Kế toán
    if any(k in text for k in ['kế toán', 'kiểm toán', 'tài chính', 'ngân hàng', 'chứng khoán', 'bảo hiểm', 'đầu tư', 'tín dụng', 'thẩm định']):
        return 'Tài chính - Kế toán'
    
    # 3. Kinh doanh - Bán hàng
    if any(k in text for k in ['bán hàng', 'kinh doanh', 'sale', 'telesale', 'thương mại điện tử', 'bán lẻ', 'bán sỉ', 'phát triển thị trường', 'showroom']):
        return 'Kinh doanh - Bán hàng'
    
    # 4. Marketing - Truyền thông
    if any(k in text for k in ['marketing', 'tiếp thị', 'quảng cáo', 'truyền thông', 'đối ngoại', 'pr', 'copywriter', 'content', 'biên tập', 'sự kiện']):
        return 'Marketing - Truyền thông'
    
    # 5. Kỹ thuật - Sản xuất
    if any(k in text for k in ['kỹ thuật', 'cơ khí', 'ô tô', 'điện', 'điện tử', 'tự động hóa', 'sản xuất', 'vận hành', 'bảo trì', 'sửa chữa', 'qa', 'qc', 'an toàn lao động']):
        return 'Kỹ thuật - Sản xuất'
    
    # 6. Xây dựng - Kiến trúc - BĐS
    if any(k in text for k in ['xây dựng', 'kiến trúc', 'nội thất', 'ngoại thất', 'bất động sản', 'nhà đất', 'địa chính', 'trắc địa', 'cầu đường']):
        return 'Xây dựng - BĐS'
    
    # 7. Dịch vụ - Du lịch - F&B
    if any(k in text for k in ['khách sạn', 'nhà hàng', 'du lịch', 'thực phẩm', 'đồ uống', 'f&b', 'pha chế', 'đầu bếp', 'phục vụ', 'spa', 'làm đẹp', 'thẩm mỹ']):
        return 'Dịch vụ - F&B - Làm đẹp'
    
    # 8. Vận tải - Logistics
    if any(k in text for k in ['vận tải', 'vận chuyển', 'logistics', 'kho vận', 'giao nhận', 'xuất nhập khẩu', 'lái xe', 'phụ xe', 'hàng hải', 'hàng không']):
        return 'Vận tải - Logistics'
    
    # 9. Y tế - Dược - Sinh học
    if any(k in text for k in ['y tế', 'dược', 'bác sĩ', 'y tá', 'điều dưỡng', 'sinh học', 'hóa học', 'mỹ phẩm']):
        return 'Y tế - Dược'
    
    # 10. Hành chính - Nhân sự - Pháp lý
    if any(k in text for k in ['nhân sự', 'hành chính', 'văn phòng', 'thư ký', 'trợ lý', 'luật', 'pháp lý', 'biên phiên dịch', 'lễ tân', 'tổng vụ']):
        return 'Hành chính - Nhân sự'
    
    # 11. Giáo dục - Đào tạo
    if any(k in text for k in ['giáo dục', 'đào tạo', 'giảng viên', 'giáo viên', 'trợ giảng', 'học vụ']):
        return 'Giáo dục - Đào tạo'
    
    # 12. Lao động phổ thông
    if any(k in text for k in ['công nhân', 'lao động phổ thông', 'giúp việc', 'tạp vụ', 'bảo vệ', 'an ninh', 'giao hàng']):
        return 'Lao động phổ thông'
    
    # 13. Nông - Lâm - Ngư nghiệp
    if any(k in text for k in ['nông nghiệp', 'lâm nghiệp', 'thủy sản', 'hải sản', 'chăn nuôi', 'thú y', 'trồng trọt']):
        return 'Nông - Lâm - Ngư nghiệp'

    return 'Khác'

In [14]:
# import re


# with open("vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
#     vietnamese_stopwords = set([line.strip() for line in f if line.strip()])
# def advanced_clean_text(text):
#     if pd.isna(text): return ""
#     text = str(text).lower()
    
#     # 1. Bảo vệ từ khóa kỹ thuật (C#, .NET...)
#     text = re.sub(r'\bc#\b', 'csharp', text)
#     text = re.sub(r'\.net\b', 'dotnet', text)
    
#     # 2. Tách dấu chấm THÔNG MINH:
#     # Chỉ tách dấu chấm nếu trước hoặc sau nó KHÔNG phải là chữ số
#     # Giúp bảo vệ 10.000.000 hoặc 3.5 năm
#     text = re.sub(r'(?<!\d)\.|\.(?!\d)', ' . ', text)
    
#     # 3. Loại bỏ các ký tự đặc biệt khác nhưng giữ lại chữ cái và số
#     text = re.sub(r'[^\w\s\.]', ' ', text)
    
#     # 4. Xóa các từ dừng
#     words = text.split()
#     words = [w for w in words if w not in vietnamese_stopwords and len(w) > 1 or w == '.']
    
#     return " ".join(words).strip()

In [15]:
import pandas as pd
import re
import numpy as np

def final_clean_salary_row(row):
    raw_min = str(row['salary_min']).lower().strip() if pd.notna(row['salary_min']) else ""
    raw_max = str(row['salary_max']).lower().strip() if pd.notna(row['salary_max']) else ""
    combined_text = raw_min + " " + raw_max

    # 1. Type 3: Thỏa thuận
    keywords = ['thỏa thuận', 'cạnh tranh', 'thương lượng', 'negotiable']
    if any(k in combined_text for k in keywords) or (raw_min == "" and raw_max == ""):
        return pd.Series([np.nan, np.nan, 3])

    # 2. Lấy toàn bộ số
    def get_nums(text):
        clean_str = text.replace('.', '').replace(',', '')
        return [float(n) for n in re.findall(r'\d+', clean_str)]

    all_nums = get_nums(combined_text)
    if not all_nums:
        return pd.Series([np.nan, np.nan, 3])

    is_usd = 'usd' in combined_text

    def calc(val):
        if is_usd:
            val *= 26335
        # Nhân 1 triệu nếu số nhỏ hoặc có từ "tr"/"triệu"
        if val < 1000 or 'triệu' in combined_text or re.search(r'\btr\b', combined_text):
            return val * 1_000_000
        return val

    # 3. Dùng \b để tránh false positive với 'từ'
    is_under = bool(re.search(r'\b(dưới|đến|tới|lên đến|upto|up to)\b', combined_text))
    is_over  = bool(re.search(r'\b(trên|hơn|tối thiểu|từ|from)\b', combined_text))

    # ⚡ Xử lý đặc biệt: "Trên X triệu" (ví dụ: "Trên 40 triệu")
    # Ưu tiên kiểm tra combined_text của từng ô riêng
    raw_over  = bool(re.search(r'\b(trên|hơn|tối thiểu|từ|from)\b', raw_min + " " + raw_max))
    raw_under = bool(re.search(r'\b(dưới|đến|tới|lên đến|upto|up to)\b', raw_min + " " + raw_max))

    f_min, f_max = np.nan, np.nan
    s_type = 0

    if len(all_nums) >= 2:
        n1, n2 = calc(all_nums[0]), calc(all_nums[1])
        f_min, f_max, s_type = min(n1, n2), max(n1, n2), 0

    elif len(all_nums) == 1:
        val = calc(all_nums[0])
        if raw_over and not raw_under:
            # "Trên 40 triệu" → min=40tr, max=NaN, type=1
            f_min, f_max, s_type = val, np.nan, 1
        elif raw_under and not raw_over:
            # "Dưới 20 triệu" → min=NaN, max=20tr, type=2
            f_min, f_max, s_type = np.nan, val, 2
        else:
            # Chỉ 1 số, không rõ hướng → coi là mức sàn
            f_min, f_max, s_type = val, np.nan, 1

    return pd.Series([f_min, f_max, s_type])

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Chuyển đổi Kinh nghiệm (Work Experience) từ chuỗi sang số
def extract_user_exp(exp_str):
    if pd.isna(exp_str): return 0, 0
    t = str(exp_str).lower()
    if 'dưới 1 năm' in t: return 0, 1
    if '1-3 năm' in t: return 1, 3
    if '3-5 năm' in t: return 3, 5
    if '5-10 năm' in t: return 5, 10
    if 'trên 10 năm' in t: return 10, 15 # Giả định trần là 15 cho GNN
    return 0, 0

df_user[['exp_min', 'exp_max']] = df_user['Work Experience'].apply(
    lambda x: pd.Series(extract_user_exp(x))
)

# 2. Xử lý Lương mong muốn (Desired Salary) -> Cần đồng nhất với Salary Job
# Trong data user, đa số là "Thỏa thuận". Ta gán nhãn salary_type = 3
def process_user_salary(df_user):
    # Tạo một dataframe tạm thời để khớp với cấu trúc hàm final_clean_salary_row
    temp_salary_df = pd.DataFrame()
    temp_salary_df['salary_min'] = df_user['Desired Salary']
    temp_salary_df['salary_max'] = np.nan # User thường không ghi mức max mong muốn

    # Áp dụng hàm của bạn đã viết
    user_salary_cleaned = temp_salary_df.apply(final_clean_salary_row, axis=1)
    
    # Đặt tên lại cho các cột kết quả
    user_salary_cleaned.columns = ['salary_min', 'salary_max', 'salary_type']
    
    # Gộp vào dataframe chính của user
    df_user = pd.concat([df_user.drop(columns=['salary_min', 'salary_max', 'salary_type'], errors='ignore'), 
                         user_salary_cleaned], axis=1)
    
    return df_user

# Thực hiện tách
df_user = process_user_salary(df_user)
# Với User, ta coi Desired Salary là mức tối thiểu họ muốn
df_user['salary_type'] = df_user['salary_min'].apply(lambda x: 3 if pd.isna(x) else 1)

# 3. Chuẩn hóa Địa điểm & Ngành nghề (Dùng chung hàm với Job)
# Cần đảm bảo bạn đã định nghĩa provinces_list và province_aliases từ trước
df_user['province'] = df_user['Workplace Desired'].apply(extract_province)
df_user['industry_group'] = df_user['Industry'].apply(map_industry_group)

# 4. Tiền xử lý văn bản (Skills + Target)
# Gộp 2 cột quan trọng nhất để tạo nội dung đặc trưng cho User
df_user["user_text"] = (
    df_user["Target"].fillna("") + " " + 
    df_user["Skills"].fillna("")
)

# Áp dụng hàm advanced_clean_text (Hàm đã bảo vệ C#, .NET... ở bước trước)
df_user["user_text_clean"] = df_user["user_text"].apply(advanced_clean_text)

# 5. Khớp nối dữ liệu số (Numerical Scaling)
# QUAN TRỌNG: Dùng bộ Scaler đã fit bên Job để transform User
# Giả sử scaler_job đã được train bên pipeline Job
# df_user[['salary_min', 'exp_min', 'exp_max']] = scaler_job.transform(df_user[['salary_min', 'exp_min', 'exp_max']])

In [18]:
# Tính median lương theo từng industry_group từ dữ liệu JOB
job_medians = df_job.groupby('industry_group')['salary_min'].median().to_dict()

# Điền vào những ô NaN của User dựa trên ngành của họ
df_user['salary_min'] = df_user['salary_min'].fillna(df_user['industry_group'].map(job_medians))
# Nếu ngành đó ở Job không có, điền median tổng
df_user['salary_min'] = df_user['salary_min'].fillna(df_job['salary_min'].median())

In [19]:
df_user.to_csv("processed/USER_DATA_PROCESSED1.csv", index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

df_job  = pd.read_csv("processed/COMBINED_DATA_PROCESSED.csv", encoding='utf-8-sig')   # file job đã xử lý xong
df_user = pd.read_csv("processed/USER_DATA_PROCESSED.csv", encoding='utf-8-sig')

# # Ghép text job
# df_job["job_text"] = (
#     df_job["job_description"].fillna("") + " " +
#     df_job["job_requirement"].fillna("")
# )

# # Ghép text user
df_user["user_text"] = (
    df_user["Skills"].fillna("") + " " +
    df_user["Target"].fillna("")
)

# ✅ Fit trên CẢ HAI corpus — đây là chỗ bug cũ chỉ có job_text
all_texts = pd.concat([
    df_job["job_text"],
    df_user["user_text"]
], ignore_index=True)

tfidf = TfidfVectorizer(
    max_features=256,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.9,
)
tfidf.fit(all_texts)

job_text_emb  = tfidf.transform(df_job["job_text"]).toarray()   # (39038, 256)
user_text_emb = tfidf.transform(df_user["user_text"]).toarray() # (3983,  256)

np.save("job_text_emb.npy",  job_text_emb)
np.save("user_text_emb.npy", user_text_emb)
print(f"Job:  {job_text_emb.shape}")
print(f"User: {user_text_emb.shape}")

Job:  (39036, 256)
User: (3983, 256)


In [ ]:
# from sklearn.preprocessing import LabelEncoder

# # 1. Khởi tạo các bộ mã hóa
# le_ind = LabelEncoder()
# le_prov = LabelEncoder()

# # 2. FIT và TRANSFORM trên dữ liệu JOB (Bước này là để xây dựng từ điển)
# df_job["industry_enc"] = le_ind.fit_transform(df_job["industry_group"].fillna("Khác"))
# df_job["province_enc"] = le_prov.fit_transform(df_job["province"].fillna("Khác"))

# # 3. CHỈ TRANSFORM trên dữ liệu USER (Dùng lại le_ind và le_prov đã fit ở trên)
# # Tuyệt đối không khởi tạo lại le_ind = LabelEncoder() ở đây
# df_user["industry_enc"] = le_ind.transform(df_user["industry_group"].fillna("Khác"))
# df_user["province_enc"] = le_prov.transform(df_user["province"].fillna("Khác"))

# # Kiểm tra thử
# print("Mã hóa ngành nghề của User:", df_user["industry_enc"].unique())

In [ ]:
from sklearn.preprocessing import RobustScaler, LabelEncoder
import numpy as np

# --- BƯỚC 1: KHỞI TẠO (Chỉ làm 1 lần) ---
scaler = RobustScaler()
le_ind = LabelEncoder()
le_prov = LabelEncoder()
le_emp = LabelEncoder()
le_func = LabelEncoder()

# --- BƯỚC 2: XỬ LÝ JOB (Học từ điển từ Job) ---
# Xử lý số
job_num_raw = df_job[["salary_min", "salary_max", "exp_min", "exp_max"]].fillna(0)
X_job_num = scaler.fit_transform(job_num_raw) # FIT và TRANSFORM

# Xử lý phân loại (Fit từ điển)
df_job["industry_enc"] = le_ind.fit_transform(df_job["industry_group"].fillna("Khác"))
df_job["province_enc"] = le_prov.fit_transform(df_job["province"].fillna("Khác"))
df_job["emp_enc"] = le_emp.fit_transform(df_job["employment_type"].fillna("full_time"))
df_job["func_enc"] = le_func.fit_transform(df_job["job_function"].fillna("Nhân viên"))

# Gộp X_job (265 chiều)
X_job = np.hstack([X_job_num, df_job[["salary_type","emp_enc","func_enc","industry_enc","province_enc"]].values, job_text_emb])


# --- BƯỚC 3: XỬ LÝ USER (Dùng lại từ điển của Job) ---
# Xử lý số (Phải đủ 4 cột và cùng tên để Scaler không báo lỗi)
user_num_raw = pd.DataFrame({
    "salary_min": df_user["salary_min"],
    "salary_max": df_user["salary_min"], # Giả định max = min cho user
    "exp_min": df_user["exp_min"],
    "exp_max": df_user["exp_max"]
}).fillna(0)
X_user_num = scaler.transform(user_num_raw) # CHỈ TRANSFORM

# Xử lý phân loại (DÙNG LẠI các le_ đã fit ở trên)
df_user["industry_enc"] = le_ind.transform(df_user["industry_group"].fillna("Khác"))
df_user["province_enc"] = le_prov.transform(df_user["province"].fillna("Khác"))
df_user["emp_enc"] = 0 # Mặc định vì User thường không có cột này
df_user["func_enc"] = 0 

# Gộp X_user (Phải ra đúng 265 chiều để khớp với Job)
X_user = np.hstack([X_user_num, df_user[["salary_type","emp_enc","func_enc","industry_enc","province_enc"]].values, user_text_emb])

In [ ]:
import torch
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from sklearn.preprocessing import LabelEncoder


# --- Xây dựng edges: User → Job (dựa trên match industry + province) ---
df_job  = df_job.reset_index(drop=True)
df_user = df_user.reset_index(drop=True)

user_ids, job_ids = [], []
for uid in range(len(df_user)):
    u = df_user.iloc[uid]
    # Positive: cùng ngành + cùng tỉnh
    mask = (
        (df_job["industry_group"] == u["industry_group"]) &
        (df_job["province"]       == u["province"])
    )
    matched = df_job[mask].index.tolist()
    # Nếu không đủ thì fallback: chỉ cùng ngành
    if len(matched) < 3:
        matched = df_job[df_job["industry_group"] == u["industry_group"]].index.tolist()
    # Lấy tối đa 10 jobs mỗi user
    sampled = np.random.choice(matched, size=min(10, len(matched)), replace=False)
    user_ids.extend([uid] * len(sampled))
    job_ids.extend(sampled.tolist())

edge_index = torch.tensor([user_ids, job_ids], dtype=torch.long)
print(f"Edges: {edge_index.shape[1]:,} (user→job)")

# --- HeteroData ---
data = HeteroData()
data["user"].x = torch.tensor(X_user, dtype=torch.float)
data["job"].x  = torch.tensor(X_job,  dtype=torch.float)
data["user", "applies", "job"].edge_index = edge_index

# --- Train/Val/Test split ---
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=False,
    edge_types=[("user", "applies", "job")],
    rev_edge_types=[("job", "rev_applies", "user")],
)
train_data, val_data, test_data = transform(data)

print(train_data)
print("Train edges:", train_data["user","applies","job"].edge_index.shape[1])
print("Val edges:  ", val_data["user","applies","job"].edge_label_index.shape[1])
print("Test edges: ", test_data["user","applies","job"].edge_label_index.shape[1])

# --- Lưu lại ---
torch.save(train_data, "train_data.pt")
torch.save(val_data,   "val_data.pt")
torch.save(test_data,  "test_data.pt")
print("Saved!")

Edges: 38,370 (user→job)
HeteroData(
  user={ x=[3983, 265] },
  job={ x=[39036, 265] },
  (user, applies, job)={
    edge_index=[2, 30696],
    edge_label=[61392],
    edge_label_index=[2, 61392],
  },
  (job, rev_applies, user)={}
)
Train edges: 30696
Val edges:   7674
Test edges:  7674
Saved!
